# YaYa AI - YouTube Shorts Generator

Turn any idea into a publish-ready YouTube Short using AI

**Features:**
- AI Script Generation (Ollama + Qwen2.5)
- Stock Video (Pexels/Pixabay)
- Text-to-Speech (Edge TTS)
- Auto Subtitles (Whisper)
- Thumbnail Generation
- 1080x1920 HD Video

---

## Enable GPU

**Important:** Go to `Run time` > `Change runtime type` > Select **GPU** (T4 x2)

This is required for faster video processing.

## Step 1: Install Dependencies

This installs FFmpeg, Ollama, and Python packages.

In [ ]:
# Install FFmpeg
!apt-get update && apt-get install -y ffmpeg

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print('Dependencies installed!')

## Step 2: Clone Repository

In [ ]:
!git clone https://github.com/madhielyousfi/YaYa-ai.git /kaggle/working/YaYa-ai
!cd /kaggle/working/YaYa-ai && pip install -r api/requirements.txt
!cd /kaggle/working/YaYa-ai && pip install typer pydantic pydantic-settings httpx edge-tts pillow python-dotenv

print('Repository cloned and dependencies installed!')

## Step 3: Configure API Keys

Get free API keys:
- **Pexels**: https://www.pexels.com/api/
- **Pixabay**: https://pixabay.com/api/docs/

Enter your keys below:

In [ ]:
#@title Enter your API keys here
PEXELS_API_KEY = "" #@param {type:"string"}
PIXABAY_API_KEY = "" #@param {type:"string"}

# Write .env file using Python (avoids shell escaping issues)
env_content = f"""PEXELS_API_KEY={PEXELS_API_KEY}
PIXABAY_API_KEY={PIXABAY_API_KEY}
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:3b
TTS_VOICE=en-US-ChristopherNeural"""

with open('/kaggle/working/YaYa-ai/.env', 'w') as f:
    f.write(env_content)

print('API keys configured!')

## Step 4: Pull Ollama Model

This downloads the Qwen2.5:3b model (~2GB).

In [ ]:
# Start Ollama in background
!ollama serve &
!sleep 5

# Pull model
!ollama pull qwen2.5:3b

print('Ollama model ready!')

## Step 5: Generate Video

Enter your topic below and run the cell!

In [ ]:
#@title Enter Your Video Topic
topic = "5 amazing facts about the ocean" #@param {type:"string"}
duration = 60 #@param {type:"integer"}
voice = "en-US-ChristopherNeural" #@param ["en-US-ChristopherNeural", "en-US-GuyNeural", "en-US-AndrewNeural", "en-US-AvaNeural", "en-US-EmmaNeural"]

import os
os.chdir('/kaggle/working/YaYa-ai')

# Update voice in .env using Python
env_path = '/kaggle/working/YaYa-ai/.env'
with open(env_path, 'r') as f:
    env_content = f.read()

import re
env_content = re.sub(r'TTS_VOICE=.*', f'TTS_VOICE={voice}', env_content)

with open(env_path, 'w') as f:
    f.write(env_content)

# Run pipeline
!python cli.py run --topic "$TOPIC" --duration $DURATION

## Step 6: View Output

Kaggle automatically downloads output files. Check the **Output** tab on the right.

In [ ]:
import os
import glob

# Find latest project
projects = sorted(glob.glob('/kaggle/working/YaYa-ai/output/*/'))
if projects:
    latest = projects[-1]
    mp4_files = glob.glob(os.path.join(latest, '*.mp4'))
    
    if mp4_files:
        video_path = mp4_files[0]
        print(f'Video ready: {video_path}')
        
        # Copy to Kaggle output
        import shutil
        output_dir = '/kaggle/working'
        dest = os.path.join(output_dir, os.path.basename(video_path))
        shutil.copy(video_path, dest)
        print(f'Saved to: {dest}')
        print('Check the Output tab on the right!')
    else:
        print('No video found. Check output folder.')
else:
    print('No projects found. Generate a video first!')

## Optional: Save to Kaggle Dataset

Save your videos as a Kaggle Dataset for easy sharing.

In [ ]:
#@title Save to Kaggle Dataset
import os
import glob
import shutil

dataset_name = "yaya-ai-videos" #@param {type:"string"}

# Find latest project
projects = sorted(glob.glob('/kaggle/working/YaYa-ai/output/*/'))
if projects:
    latest = projects[-1]
    dest = f'/kaggle/working/{dataset_name}'
    os.makedirs(dest, exist_ok=True)
    
    # Copy all files
    for item in os.listdir(latest):
        src = os.path.join(latest, item)
        dst = os.path.join(dest, item)
        if os.path.isfile(src):
            shutil.copy(src, dst)
        elif os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    
    print(f'Saved to dataset: {dest}')
    print('To publish: Go to Output tab > New Dataset')
else:
    print('No projects to save.')